References

- https://www.kaggle.com/code/richolson/ai-math-olympiad-qwen2-5-72b for showing how to submit
- https://www.kaggle.com/code/abdullahmeda/load-72b-awq-model-using-vllm-on-l4-x4
- https://www.kaggle.com/code/huikang/qwen2-5-math-1-5b-instruct

In [1]:
import os
import gc
import time
import warnings

import pandas as pd
import polars as pl

import torch
import kaggle_evaluation.aimo_2_inference_server

pd.set_option('display.max_colwidth', None)
cutoff_time = time.time() + (4 * 60 + 30) * 60

In [2]:
from vllm import LLM, SamplingParams

warnings.simplefilter('ignore')

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def clean_memory(deep=False):
    gc.collect()
    if deep:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    torch.cuda.empty_cache()


llm_model_pth = '/kaggle/input/qwen2.5/transformers/72b-instruct-awq/1'

llm = LLM(
    llm_model_pth,
    dtype="half",                # The data type for the model weights and activations
    max_num_seqs=8,              # Maximum number of sequences per iteration. Default is 256
    max_model_len=4096,          # Model context length
    trust_remote_code=True,      # Trust remote code (e.g., from HuggingFace) when downloading the model and tokenizer
    tensor_parallel_size=4,      # The number of GPUs to use for distributed execution with tensor parallelism
    gpu_memory_utilization=0.95, # The ratio (between 0 and 1) of GPU memory to reserve for the model
    seed=2024,
)

2024-11-05 07:33:29,126	INFO util.py:124 -- Outdated packages:
  ipywidgets==7.7.1 found, needs ipywidgets>=8
Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 11-05 07:33:58 awq_marlin.py:97] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 11-05 07:33:58 config.py:905] Defaulting to use mp for distributed inference
INFO 11-05 07:33:58 llm_engine.py:237] Initializing an LLM engine (v0.6.3.post1) with config: model='/kaggle/input/qwen2.5/transformers/72b-instruct-awq/1', speculative_config=None, tokenizer='/kaggle/input/qwen2.5/transformers/72b-instruct-awq/1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=4, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=awq_marlin, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_confi

Loading safetensors checkpoint shards:   0% Completed | 0/11 [00:00<?, ?it/s]


(VllmWorkerProcess pid=356) INFO 11-05 07:38:44 model_runner.py:1067] Loading model weights took 9.7875 GB
INFO 11-05 07:38:44 model_runner.py:1067] Loading model weights took 9.7875 GB
(VllmWorkerProcess pid=357) INFO 11-05 07:38:44 model_runner.py:1067] Loading model weights took 9.7875 GB
(VllmWorkerProcess pid=355) INFO 11-05 07:38:44 model_runner.py:1067] Loading model weights took 9.7875 GB
INFO 11-05 07:38:51 distributed_gpu_executor.py:57] # GPU blocks: 8633, # CPU blocks: 3276
INFO 11-05 07:38:51 distributed_gpu_executor.py:61] Maximum concurrency for 4096 tokens per request: 33.72x
INFO 11-05 07:38:55 model_runner.py:1395] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
(VllmWorkerProcess pid=356) INFO 11-05 07:38:55 model_runner.py:1399] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory,

In [3]:
tokenizer = llm.get_tokenizer()

In [4]:
import re
import keyword


def extract_python_code(text):
    pattern = r'```python\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)
    return "\n\n".join(matches)


def process_python_code(query):
    # Add import statements
    # Also print variables if they are not inside any indentation
    query = "import math\nimport numpy as np\nimport sympy as sp\n" + query
    current_rows = query.strip().split("\n")
    new_rows = []
    for row in current_rows:
        new_rows.append(row)
        if not row.startswith(" ") and "=" in row:
            variables_to_print = row.split("=")[0].strip()
            for variable_to_print in variables_to_print.split(","):
                variable_to_print = variable_to_print.strip()
                if variable_to_print.isidentifier() and not keyword.iskeyword(variable_to_print):
                    if row.count("(") == row.count(")") and row.count("[") == row.count("]"):
                        # TODO: use some AST to parse code
                        new_rows.append(f'\ntry:\n    print(f"{variable_to_print}={{str({variable_to_print})[:100]}}")\nexcept:\n    pass\n')
    return "\n".join(new_rows)


def extract_boxed_text(text):
    pattern = r'oxed{(.*?)}'
    matches = re.findall(pattern, text)
    if not matches:
        return ""
    return matches[0]


from collections import Counter
import random
def select_answer(answers):
    counter = Counter()
    for answer in answers:
        try:
            if int(answer) == float(answer):
                counter[int(answer)] += 1 + random.random() / 1_000
        except:
            pass
    if not counter:
        return 210
    _, answer = sorted([(v,k) for k,v in counter.items()], reverse=True)[0]
    return answer%1000

In [5]:
import os
import tempfile
import subprocess

class PythonREPL:
    def __init__(self, timeout=5):
        self.timeout = timeout

    def __call__(self, query):
        with tempfile.TemporaryDirectory() as temp_dir:
            temp_file_path = os.path.join(temp_dir, "tmp.py")
            with open(temp_file_path, "w", encoding="utf-8") as f:
                f.write(query)
            
            try:
                result = subprocess.run(
                    ["python3", temp_file_path],
                    capture_output=True,
                    check=False,
                    text=True,
                    timeout=self.timeout,
                )
            except subprocess.TimeoutExpired:
                return False, f"Execution timed out after {self.timeout} seconds."

            stdout = result.stdout.strip()
            stderr = result.stderr.strip()

            if result.returncode == 0:
                return True, stdout
            else:
                # Process the error message to remove the temporary file path
                # This makes the error message cleaner and more user-friendly
                error_lines = stderr.split("\n")
                cleaned_errors = []
                for line in error_lines:
                    if temp_file_path in line:
                        # Remove the path from the error line
                        line = line.replace(temp_file_path, "<temporary_file>")
                    cleaned_errors.append(line)
                cleaned_error_msg = "\n".join(cleaned_errors)
                # Include stdout in the error case
                combined_output = f"{stdout}\n{cleaned_error_msg}" if stdout else cleaned_error_msg
                return False, combined_output

In [6]:
sampling_params = SamplingParams(
    temperature=1.0,              # randomness of the sampling
    min_p=0.01,
    skip_special_tokens=True,     # Whether to skip special tokens in the output.
    max_tokens=2400,
    stop=["```\n"],
    include_stop_str_in_output=True,
)

def batch_message_generate(list_of_messages) -> list[list[dict]]:

    list_of_texts = [
        tokenizer.apply_chat_template(
            conversation=messages,
            tokenize=False,
            add_generation_prompt=True
        )
        for messages in list_of_messages
    ]

    request_output = llm.generate(
        prompts=list_of_texts,
        sampling_params=sampling_params,
    )
    
    for messages, single_request_output in zip(list_of_messages, request_output):
        # print()
        # print(single_request_output.outputs[0].text)
        # print()
        messages.append({'role': 'assistant', 'content': single_request_output.outputs[0].text})

    return list_of_messages

In [7]:
def batch_message_filter(list_of_messages) -> tuple[list[list[dict]], list[str]]:
    extracted_answers = []
    list_of_messages_to_keep = []
    for messages in list_of_messages:
        answer = extract_boxed_text(messages[-1]['content'])
        if answer:
            extracted_answers.append(answer)
        else:
            list_of_messages_to_keep.append(messages)
    return list_of_messages_to_keep, extracted_answers

In [8]:
def batch_message_execute(list_of_messages) -> list[list[dict]]:
    for messages in list_of_messages:
        python_code = extract_python_code(messages[-1]['content'])
        python_code = process_python_code(python_code)
        # print('\n\n' + python_code + '\n\n')
        try:
            print('c', end='')
            is_successful, output = PythonREPL()(python_code)
            if is_successful:
                print('o', end='')
            else:
                print('e', end='')
        except Exception as e:
            print('f', end='')
            output = str(e)
        print(python_code)
        print()
        print(output)
        print("\n\n")
        messages.append({'role': 'user', 'content': output})
    print()
    return list_of_messages

In [9]:
import os

import pandas as pd
import polars as pl

import kaggle_evaluation.aimo_2_inference_server

In [10]:
def create_starter_messages(question, index):
    cycle_size = 2
    if False:
        pass
    elif index % cycle_size == 1:
        # https://github.com/QwenLM/Qwen2.5-Math?tab=readme-ov-file#-hugging-face-transformers
        return [
            {"role": "system", "content": "Please reason step by step, and put your final answer within \\boxed{}."},
            {"role": "user", "content": question}
        ]
    else:
        # https://github.com/QwenLM/Qwen2.5-Math?tab=readme-ov-file#-hugging-face-transformers
        return [
            {"role": "system", "content": "Please integrate natural language reasoning with programs to solve the problem above, and put your final answer within \\boxed{}."},
            {"role": "user", "content": question + "\n\nBegin your answer by importing sympy."}
        ]

In [11]:
def predict_for_question(question: str) -> int:
    import os
    if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
        if question != "Triangle $ABC$ has side length $AB = 120$ and circumradius $R = 100$. Let $D$ be the foot of the perpendicular from $C$ to the line $AB$. What is the greatest possible length of segment $CD$?":
            return 210
    if time.time() > cutoff_time:
        return 210
    
    question += "\nIf the final answer is a number larger than 1 million, take modulo 1000."
    print(question)

    list_of_messages = [create_starter_messages(question, index) for index in range(32)]

    all_extracted_answers = []
    for _ in range(4):
        list_of_messages = batch_message_generate(list_of_messages)
        list_of_messages, extracted_answers = batch_message_filter(list_of_messages)
        all_extracted_answers.extend(extracted_answers)
        if not list_of_messages:
            break
        list_of_messages = batch_message_execute(list_of_messages)

    print(all_extracted_answers)
    answer = select_answer(all_extracted_answers)
    print(answer)

    print("\n\n")
    return answer

In [12]:
# Replace this function with your inference code.
# The function should return a single integer between 0 and 999, inclusive.
# Each prediction (except the very first) must be returned within 30 minutes of the question being provided.
def predict(id_: pl.DataFrame, question: pl.DataFrame) -> pl.DataFrame | pd.DataFrame:
    id_ = id_.item(0)
    print("------")
    print(id_)
    
    question = question.item(0)
    answer = predict_for_question(question)
    print(question)
    print("------\n\n\n")
    return pl.DataFrame({'id': id_, 'answer': answer})

In [13]:
# predict_for_question("Triangle $ABC$ has side length $AB = 120$ and circumradius $R = 100$. Let $D$ be the foot of the perpendicular from $C$ to the line $AB$. What is the greatest possible length of segment $CD$?")

In [14]:
pd.read_csv(
    '/kaggle/input/ai-mathematical-olympiad-progress-prize-2/reference.csv'
).drop('answer', axis=1).to_csv('reference.csv', index=False)

In [15]:
inference_server = kaggle_evaluation.aimo_2_inference_server.AIMO2InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        (
#             '/kaggle/input/ai-mathematical-olympiad-progress-prize-2/test.csv',
            'reference.csv',
        )
    )

------
1fce4b
Find the three-digit number $n$ such that writing any other three-digit number $10^{2024}$ times in a row and $10^{2024}+2$ times in a row results in two numbers divisible by $n$.
------



------
88c219
For positive integers $x_1,\ldots, x_n$ define $G(x_1, \ldots, x_n)$ to be the sum of their $\frac{n(n-1)}{2}$ pairwise greatest common divisors. We say that an integer $n \geq 2$ is \emph{artificial} if there exist $n$ different positive integers $a_1, ..., a_n$ such that 
\[a_1 + \cdots + a_n = G(a_1, \ldots, a_n) +1.\]
Find the sum of all artificial integers $m$ in the range $2 \leq m \leq 40$.
------



------
bbd91e
Alice writes all positive integers from $1$ to $n$ on the board for some positive integer $n \geq 11$. Bob then erases ten of them. The mean of the remaining numbers is $3000/37$. The sum of the numbers Bob erased is $S$. What is the remainder when $n \times S$ is divided by $997$?
------



------
192e23
Fred and George take part in a tennis tournament w

Processed prompts: 100%|██████████| 32/32 [02:56<00:00,  5.51s/it, est. speed input: 20.80 toks/s, output: 108.76 toks/s]


coimport math
import numpy as np
import sympy as sp
import sympy as sp

# Given values
AB = 120

try:
    print(f"AB={str(AB)[:100]}")
except:
    pass

R = 100

try:
    print(f"R={str(R)[:100]}")
except:
    pass


# Let's denote the angle at C as θ
theta = sp.symbols('theta')

try:
    print(f"theta={str(theta)[:100]}")
except:
    pass


# Using the circumradius formula R = (AB * BC * CA) / (4 * Area)
# We can express the area of triangle ABC in terms of R and theta
# Area = (1/2) * AB * CD
# We also know that in a right triangle, CD = R * sin(θ)

try:
    print(f"CD={str(CD)[:100]}")
except:
    pass

CD = R * sp.sin(theta)

try:
    print(f"CD={str(CD)[:100]}")
except:
    pass


# The length of AB is fixed, so we need to maximize CD
# The maximum value of sin(θ) is 1, which occurs when θ = 90 degrees or π/2 radians
max_CD = CD.subs(theta, sp.pi/2)

try:
    print(f"max_CD={str(max_CD)[:100]}")
except:
    pass


# Simplify the expression
max_CD = sp.simplify(max_CD)

try:
    pr

Processed prompts: 100%|██████████| 16/16 [01:04<00:00,  4.00s/it, est. speed input: 141.95 toks/s, output: 79.37 toks/s]


coimport math
import numpy as np
import sympy as sp
import sympy as sp

# Given values
AB = 120

try:
    print(f"AB={str(AB)[:100]}")
except:
    pass

R = 100

try:
    print(f"R={str(R)[:100]}")
except:
    pass


# Calculate sin(C) using the Law of Sines
sin_C = AB / (2 * R)

try:
    print(f"sin_C={str(sin_C)[:100]}")
except:
    pass


# Calculate the maximum possible value of CD
CD_max = AB * sin_C

try:
    print(f"CD_max={str(CD_max)[:100]}")
except:
    pass


# If the value is larger than 1 million, take modulo 1000
if CD_max > 1000000:
    CD_max = CD_max % 1000

CD_max

AB=120
R=100
sin_C=0.6
CD_max=72.0



coimport math
import numpy as np
import sympy as sp
import sympy as sp

# Given values
AB = 120

try:
    print(f"AB={str(AB)[:100]}")
except:
    pass

R = 100

try:
    print(f"R={str(R)[:100]}")
except:
    pass


# Calculate sin(C)
sin_C = AB / (2 * R)

try:
    print(f"sin_C={str(sin_C)[:100]}")
except:
    pass


# Calculate cos(C)
cos_C = sp.sqrt(1 - sin_C**2)

t

Processed prompts: 100%|██████████| 4/4 [00:27<00:00,  6.93s/it, est. speed input: 162.64 toks/s, output: 22.29 toks/s]


coimport math
import numpy as np
import sympy as sp
import sympy as sp

# Define the symbols
x, CD = sp.symbols('x CD')

try:
    print(f"x={str(x)[:100]}")
except:
    pass


try:
    print(f"CD={str(CD)[:100]}")
except:
    pass


# Given values
AB = 120

try:
    print(f"AB={str(AB)[:100]}")
except:
    pass

R = 100

try:
    print(f"R={str(R)[:100]}")
except:
    pass


# Expressions for AC and BC using the Pythagorean theorem
AC_squared = x**2 + CD**2

try:
    print(f"AC_squared={str(AC_squared)[:100]}")
except:
    pass

BC_squared = (AB - x)**2 + CD**2

try:
    print(f"BC_squared={str(BC_squared)[:100]}")
except:
    pass


# Substitute AC and BC into the circumradius formula
# (AC * BC) = 2 * R * CD
# (AC^2 * BC^2) = (2 * R * CD)^2
equation = (AC_squared * BC_squared) - (2 * R * CD)**2

try:
    print(f"equation={str(equation)[:100]}")
except:
    pass


# Solve the equation for CD
CD_solutions = sp.solve(equation, CD)

try:
    print(f"CD_solutions={str(CD_solutions)[:100]}

Processed prompts: 100%|██████████| 1/1 [00:21<00:00, 21.64s/it, est. speed input: 111.22 toks/s, output: 17.00 toks/s]


coimport math
import numpy as np
import sympy as sp
import sympy as sp

# Define the symbols
x = sp.symbols('x')

try:
    print(f"x={str(x)[:100]}")
except:
    pass


# Given values
AB = 120

try:
    print(f"AB={str(AB)[:100]}")
except:
    pass

R = 100

try:
    print(f"R={str(R)[:100]}")
except:
    pass


# Define the solutions for CD
CD_solutions = [
    80 - sp.sqrt(-x**2 + 120*x + 6400),
    -sp.sqrt(-x**2 + 120*x + 6400) - 80,
    sp.sqrt(-x**2 + 120*x + 6400) - 80
]

# Function to evaluate the maximum CD
def find_max_CD(CD_solutions):
    max_CD = 0
    for x_val in range(121):  # x ranges from 0 to 120
        for sol in CD_solutions:
            CD_val = sol.subs(x, x_val).evalf()
            if CD_val.is_real and CD_val > max_CD:
                max_CD = CD_val
    return max_CD

# Find the maximum CD
max_CD = find_max_CD(CD_solutions)

try:
    print(f"max_CD={str(max_CD)[:100]}")
except:
    pass


# Check if the answer is larger than 1 million and take modulo 1000 if 